In [1]:
import os, io, base64, numpy as np, requests, torch, torch.nn as nn
from typing import Tuple
from PIL import Image

BASE_URL = "http://target_ip:target_port"
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def x01_from_b64_png(b64: str) -> np.ndarray:
    """Convert base64 PNG to [0,1] numpy array.

    Args:
        b64: Base64 encoded PNG string

    Returns:
        np.ndarray: Image as (28, 28) array in [0,1] range
    """
    raw = base64.b64decode(b64)
    img = Image.open(io.BytesIO(raw)).convert("L")
    if img.size != (28, 28):
        raise ValueError("Expected 28x28 PNG")
    x = np.asarray(img, dtype=np.float32) / 255.0
    return np.clip(x, 0.0, 1.0)

def b64_png_from_x01(x2d: np.ndarray) -> str:
    """Convert [0,1] array to base64 PNG.

    Args:
        x2d: Image array in [0,1] range

    Returns:
        str: Base64 encoded PNG string
    """
    x255 = np.clip((x2d * 255.0).round(), 0, 255).astype(np.uint8)
    img = Image.fromarray(x255, mode="L")
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    return base64.b64encode(buf.getvalue()).decode("ascii")

def l2(a: np.ndarray, b: np.ndarray) -> float:
    """Compute L2 distance between two arrays.

    Args:
        a, b: Arrays to compare

    Returns:
        float: Euclidean distance
    """
    return float(np.linalg.norm((a - b).ravel(), ord=2))

# Fetch challenge and check clean prediction
ch = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x = x01_from_b64_png(ch["image_b64"])      # (28, 28)
lab = int(ch["label"])                     # baseline label
tgt = int(ch["target"])                    # target label
thr = float(ch["l2_threshold"])            # numeric threshold
res = requests.post(f"{BASE_URL}/predict", json={"image_b64": b64_png_from_x01(x)}, timeout=10).json()
print({"baseline_label": lab, "target": tgt, "server_pred": res["pred"], "l2_threshold": thr})

{'baseline_label': 4, 'target': 6, 'server_pred': 4, 'l2_threshold': 0.75}


In [2]:
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x01: torch.Tensor) -> torch.Tensor:
        """Forward pass with internal normalization.

        Args:
            x01: Input tensor in [0,1] with shape (N, 1, 28, 28)

        Returns:
            Log-probabilities with shape (N, 10)
        """
        x = (x01 - MNIST_MEAN) / MNIST_STD
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

# Download and load weights
wt = requests.get(f"{BASE_URL}/weights", timeout=10).content
open("deepfool_weights.pth", "wb").write(wt)

model = SimpleClassifier().eval()
state = torch.load("deepfool_weights.pth", map_location=device)
model.load_state_dict(state)

# Verify model works locally
x_tensor = torch.from_numpy(x[None, None, ...]).float()
logits = model(x_tensor)
local_pred = int(torch.argmax(logits, dim=1).item())
print(f"Local prediction: {local_pred}, should match server: {res['pred']}")

Local prediction: 4, should match server: 4


In [3]:
bad = requests.post(f"{BASE_URL}/submit", json={"image_b64": b64_png_from_x01(x)}, timeout=10)
print(bad.status_code, bad.text)  # expected: 400 with "Wrong target: predicted X, need 6"

400 {"detail":"Wrong target: predicted 4, need 6"}


In [4]:
def deepfool(image: torch.Tensor,
             target_class: int,
             net: nn.Module,
             num_classes: int = 10,
             overshoot: float = 0.02,
             max_iter: int = 50,
             device: str = 'cuda') -> Tuple[torch.Tensor, int, int, int, torch.Tensor]:
    """
    Generate minimal adversarial perturbation using DeepFool algorithm.

    Args:
        image (torch.Tensor): Input image tensor of shape (1, C, H, W)
        net (nn.Module): Target neural network in evaluation mode
        num_classes (int): Number of top-scoring classes to consider (default: 10)
        overshoot (float): Overshoot parameter for boundary crossing (default: 0.02)
        max_iter (int): Maximum iterations before terminating (default: 50)
        device (str): Computation device ('cuda' or 'cpu')

    Returns:
        Tuple containing:
            - r_tot (torch.Tensor): Total accumulated perturbation
            - loop_i (int): Number of iterations performed
            - label (int): Original predicted class
            - k_i (int): Final adversarial class
            - pert_image (torch.Tensor): Final perturbed image
    """
    image = image.to(device)
    net = net.to(device)

    # Original prediction and class ordering (descending score)
    f_image = net(image).data.cpu().numpy().flatten()
    I = f_image.argsort()[::-1]
    label = I[0]

    # Working tensors and accumulators
    input_shape = image.shape
    pert_image = image.clone()
    r_tot = torch.zeros(input_shape).to(device)
    loop_i = 0

    # Iterate until a successful perturbation is found or the limit is reached
    while loop_i < max_iter:
        x = pert_image.clone().requires_grad_(True)
        fs = net(x)
        # Current top prediction at x
        k_i = fs.data.cpu().numpy().flatten().argsort()[::-1][0]

        # Stop when the prediction changes to target class
        if k_i == target_class:
            orig_image_np = image.clone().detach().cpu().numpy().squeeze()
            pert_image_np = x.clone().detach().cpu().numpy().squeeze()
            l2_attack = l2(orig_image_np, pert_image_np)
            if(l2_attack >= thr*0.98): # Stop only if we are close to l2 norm treshhold 
                break

        # Initialize the best candidate step for this iteration
        pert = float('inf')
        w = None

        # Search minimal step among candidate classes
        for k in range(1, num_classes):
            if I[k] != target_class: # Search minimal step only for target class
                continue

            # Compute gradient for candidate class
            if x.grad is not None:
                x.grad.zero_()
            fs[0, I[k]].backward(retain_graph=True)
            grad_k = x.grad.data.clone()
    
            # Compute gradient for original class
            if x.grad is not None:
                x.grad.zero_()
            fs[0, label].backward(retain_graph=True)
            grad_label = x.grad.data.clone()
    
            # Direction and distance under linearization
            w_k = grad_k - grad_label
            f_k = (fs[0, I[k]] - fs[0, label]).data.cpu().numpy()
            pert_k = abs(f_k) / (torch.norm(w_k.flatten()) + 1e-10)
    
            if pert_k < pert:
                pert = pert_k
                w = w_k
    
            # Minimal step for the selected direction
            r_i = (pert + 1e-4) * w / (torch.norm(w.flatten()) + 1e-10)
            r_tot = r_tot + r_i
    
            # Apply with overshoot to ensure crossing
            pert_image = torch.clamp(image + (1 + overshoot) * r_tot, 0, 1)
            loop_i += 1

    return r_tot, loop_i, label, k_i, pert_image

In [5]:
r_total, iterations, orig_label, pert_label, pert_image = deepfool(
    x_tensor, tgt, model, num_classes=10, overshoot=0.02, max_iter=50, device=device
)

print(pert_label)

6


In [6]:
image_np = pert_image.detach().cpu().numpy()
image_np = image_np.squeeze()

x_tensor = torch.from_numpy(image_np[None, None, ...]).float().to(device)
logits = model(x_tensor)
local_pred = int(torch.argmax(logits, dim=1).item())
print(f"Local prediction: {local_pred}")

quantized = (image_np * 255).astype(np.uint8).astype(np.float32) / 255
q_tensor = torch.from_numpy(quantized[None, None, ...]).float().to(device)
q_logits = model(q_tensor)
q_pred = int(torch.argmax(q_logits, dim=1).item())
print(f"Local prediction on quantized image: {q_pred}")

result = requests.post(f"{BASE_URL}/predict", json={"image_b64": b64_png_from_x01(image_np)}, timeout=10).json()
print(result)

result = requests.post(f"{BASE_URL}/submit", json={"image_b64": b64_png_from_x01(image_np)}, timeout=10).json()
print(result)

Local prediction: 6
Local prediction on quantized image: 6
{'pred': 6, 'confidence': 0.41301193833351135}
{'ok': True, 'flag': 'HTB{d33pf00l_f00lz}', 'pred': 6, 'target': 6, 'l2': 0.739244282245636}
